## DynamoDB operations & DAX

Four read shapes with very different cost. **`GetItem`** — one item by primary key; cheapest and fastest. **`Query`** — items within **one PK value**, optionally filtered by SK range; efficient (reads only that partition) — the **workhorse**. **`Scan`** — reads the **entire table** then filters, **charging for the whole table** regardless of matches — almost always wrong in prod (if you're reaching for Scan, you probably want a GSI). **`BatchGetItem`/`BatchWriteItem`** — up to 100 gets / 25 writes per call, saving round-trips (not capacity). One trap: **filter expressions apply *after* the read**, so filtered-out items still cost capacity — they trim the payload, not the bill.

**DAX (DynamoDB Accelerator)** is an in-memory cache purpose-built for DynamoDB — **microsecond reads** on cache hits (vs single-digit-ms), **drop-in** with the DynamoDB SDK (swap the client, code unchanged). It's **write-through** (writes go to DynamoDB first, then update DAX), has separate **item cache** (5-min TTL) and **query cache** (1-min TTL), and runs **VPC-only, multi-AZ** as a node cluster. Reach for DAX when a workload is **read-heavy on hot items** and needs microsecond latency — not for write-heavy or strongly-consistent-read needs (DAX serves eventually-consistent).